# 14 — PTA deliverable: build-up tests → reservoir properties, plots, Excel

**Pipeline** (`src/pta.py → analyze_all`), fully automatic:
1. **Detect shut-ins**: choke opening = 0 %.
2. **Classify** each shut-in (textbook definition, Earlougher / Horne): a *pressure build-up test* is a designed
   test — stabilised production, deliberate shut-in of planned length, controlled return. Signatures used:
   **duration ≥ 24 h** and a **staged choke ramp-up** when reopened. Other closures are *unplanned shut-ins*
   (trips / interruptions): analysed and shown, but not reported as test results.
3. **Rate** `q` = mean VFM oil rate of the 7 days before the shut-in; **Pwf(Δt=0)** = mean downhole pressure of
   the last 24 h (averaged, stabilised flowing conditions).
4. **Horner analysis** on the per-minute build-up: Bourdet derivative → radial-flow window (flat derivative)
   → straight line `Pws = P* − m·log10((tp+Δt)/Δt)`, `tp = 24·Np/q`.
5. **Properties**: `k = 162.6·q·B·μ/(m·h)`, `s = 1.151·[(P1hr − Pwf)/m − log10(k/(φ·μ·ct·rw²)) + 3.23]`,
   `ΔP_skin = 0.869·m·s` (field units: STB/d, rb/STB, cp, ft, psi, md).
6. **Cross-check**: analytical model matching (storage + skin + boundary) and an automatic interpretation.

Outputs: `outputs/PTA_results.xlsx`, plots in `outputs/pta_plots/` (unplanned shut-ins in a subfolder).

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import export
import load
import pta
import pta_model

OUT = load.ROOT / 'outputs'
hf = load.load_high_freq_clean()
choke = load.load_choke_events()
pvt = load.load_pvt().set_index('well_name')
completion = load.load_completion().set_index('well_name')
vfm_daily = pd.read_excel(OUT / 'VFM_predictions.xlsx', sheet_name='predictions')   # final VFM
vfm_daily['date'] = pd.to_datetime(vfm_daily.date)
rates = vfm_daily.rename(columns={'oil_rate_stbd': 'oil'})[['date', 'well_name', 'oil']]

results = pta.analyze_all(hf, pta.find_shut_ins(hf), rates, pvt, completion, choke)
pd.DataFrame([{k: r[k] for k in ['well_name', 't_shut', 'duration_hr', 'event_type', 'classification_reason']}
              for r in results])

## Model-match cross-check, interpretation and sensitivity for every shut-in

In [ ]:
rows, sens_all, fits_all = [], [], {}
for r in results:
    p = pvt.loc[r['well_name']]
    fits, best, dt_fit, dp_fit = pta_model.fit_all_models(r, p)
    fits_all[(r['well_name'], r['t_shut'])] = (fits, best, dt_fit, dp_fit)
    event = {'well_name': r['well_name'], 't_shut': r['t_shut'], 't_open': r['t_open'], 'duration_hr': r['duration_hr']}
    sens = pta.sensitivity(hf, event, r['q_oil_stbd'], r['cum_oil_stb'], p)
    sens_all.append(sens)
    method_cases = sens[~sens.case.str.startswith('q ')]
    row = {k: v for k, v in r.items() if not k.startswith('_')}
    row.update({
        'model_boundary': best['boundary'], 'model_k_md': best['k_md'], 'model_skin': best['skin'],
        'model_C_bbl_psi': best['C_bbl_psi'], 'model_boundary_radius_ft': best['boundary_radius_ft'],
        'model_fit_rms_pct': best['rms_error_pct'],
        'k_uncertainty_analysis_pct': method_cases.k_change_pct.abs().max(),
        'skin_uncertainty_analysis': method_cases.skin_change.abs().max(),
        'interpretation': pta.interpret(r['k_md'], r['skin'], r['dP_skin_psi'], r['flow_efficiency'],
                                        best['boundary'], best['boundary_radius_ft'], best['radius_investigation_ft']),
    })
    rows.append(row)

details = pd.DataFrame(rows)
for _, r in details.iterrows():
    print(f"{r.well_name} {r.t_shut:%Y-%m-%d} [{r.event_type}]: {r.interpretation}")

## Required results: one row per PLANNED build-up test

In [ ]:
tests = details[details.event_type == 'planned build-up test']
results_table = export.pta_table(tests)
results_table

In [ ]:
all_shut_ins = details[['well_name', 't_shut', 'duration_hr', 'event_type', 'classification_reason', 'q_oil_stbd',
                        'm_psi_cycle', 'k_md', 'skin', 'dP_skin_psi', 'model_k_md', 'model_skin', 'interpretation']]
method = [
    ('shut-in detection', 'choke opening = 0 % (choke_events.csv) on the per-minute downhole pressure'),
    ('classification', 'planned build-up test = duration >= 24 h AND staged choke ramp-up on reopening; '
                       'otherwise unplanned shut-in (shown, not reported as a test result)'),
    ('rate q', 'mean VFM-predicted oil rate of the 7 days before the shut-in'),
    ('flowing pressure Pwf(dt=0)', 'mean downhole pressure of the 24 h before the shut-in'),
    ('producing time', 'tp = 24 * Np / q, Np = cum oil to 1 Jan 2025 + VFM oil since [h]'),
    ('radial flow window', 'automatic: flattest Bourdet derivative (vs Agarwal equivalent time) after the storage hump'),
    ('Horner slope m', 'Pws = P* - m * log10((tp + dt) / dt), least squares on the radial-flow minutes [psi/cycle]'),
    ('permeability', 'k = 162.6 * q * Bo * mu / (m * h) [md]'),
    ('skin', 's = 1.151 * ((P1hr - Pwf)/m - log10(k / (phi * mu * ct * rw^2)) + 3.23)'),
    ('delta P skin', 'dP_skin = 0.869 * m * s [psi]'),
    ('flow efficiency', 'FE = (P* - Pwf - dP_skin) / (P* - Pwf)'),
    ('model match (cross-check)', 'Laplace solution: storage + skin + infinite / constant-pressure / closed circle, '
                                  'Stehfest inversion, least squares on log(dP)'),
]
path = OUT / 'PTA_results.xlsx'
with pd.ExcelWriter(path, engine='openpyxl') as xl:
    results_table.to_excel(xl, sheet_name='pta_results', index=False)
    all_shut_ins.to_excel(xl, sheet_name='all_shut_ins', index=False)
    details.to_excel(xl, sheet_name='pta_details', index=False)
    pd.concat(sens_all).round(3).to_excel(xl, sheet_name='sensitivity', index=False)
    pd.DataFrame(method, columns=['item', 'value']).to_excel(xl, sheet_name='method', index=False)
print('written:', path)
all_shut_ins.round(2)

## PTA plots folder: history · Horner · log-log derivative, and the model match, per shut-in

In [ ]:
plot_dir = OUT / 'pta_plots'
(plot_dir / 'unplanned_shut_ins').mkdir(parents=True, exist_ok=True)
for r in results:
    folder = plot_dir if r['event_type'] == 'planned build-up test' else plot_dir / 'unplanned_shut_ins'
    name = f"{r['well_name']}_{r['t_shut']:%Y%m%d}"
    fig = pta.plot_event(hf, r, vfm_daily)
    fig.suptitle(f"{r['well_name']} {r['t_shut']:%Y-%m-%d %H:%M} — {r['event_type']} ({r['duration_hr']:.1f} h)", fontsize=13)
    fig.savefig(folder / f'{name}_buildup.png', dpi=120)
    plt.show()
    fits, best, dt_fit, dp_fit = fits_all[(r['well_name'], r['t_shut'])]
    fig = pta_model.plot_model_match(r, fits, best, dt_fit, dp_fit)
    fig.savefig(folder / f'{name}_model_match.png', dpi=120)
    plt.show()
sorted(str(p.relative_to(plot_dir)) for p in plot_dir.rglob('*.png'))